# Proportions by FACS (Non-EEC) & scRNA-seq (EEC)

In [ ]:
%autosave 0

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Silence warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)


# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append(str(base_proj_dir))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from dynaconf import Dynaconf
from tqdm import tqdm
from functools import reduce
import numpy as np
import pandas as pd
import numpy as np
import marsilea as ma 
import marsilea.plotter as mp 

# ---- Scanpy global settings ----
sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(
    dpi=150,
    dpi_save=300,
    format="pdf",
    facecolor="white",
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(4, 4),
    transparent=True,
)

# ---- Matplotlib global settings ----
mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
anndata_dir = Path(settings.IO.anndata)
external_data_dir = Path(settings.IO.external_data)
figures_dir = Path(settings.ANALYSIS.figures_dir)
tables_dir = Path(settings.ANALYSIS.tables_dir)
models_dir = Path(settings.ANALYSIS.models_dir)

## Load FACS data

In [ ]:
# Define color palette for each cell type
cell_type_colors = {
    "Stem_TA_Cells": "#2077b4",  
    "Enterocytes": "#2d9e48",     
    "Goblet_Cells": "#f27e20",   
    "EECs": "#9068aa",         
}

In [ ]:
# Load FACS data
facs_df = pd.read_csv(external_data_dir / "flow_quant_tf_perturbation.csv")

# Map batch date array([17072025,  1072025, 16072025, 14082025]) to 17.07.2025, 01.07.2025, 16.07.2025, 14.08.2025
# First map these wrongly spelled batch dates to correct ones
batch_date_mapping = {
    17072025: "17.07.2025",
    1072025: "01.07.2025",
    16072025: "16.07.2025",
    14082025: "14.08.2025",
}
facs_df["Batch date"] = facs_df["Batch date"].replace(batch_date_mapping)

# Now convert each batch date to datetime
facs_df["Batch date"] = pd.to_datetime(facs_df["Batch date"], errors="coerce")

# Create replicate for each Transcription factor based on Transcription factor / Batch date combination
facs_df["Replicate"] = facs_df["Transcription factor"].astype(str) + " / " + facs_df["Sample no."].astype(str) +  " / " + facs_df["Batch date"].dt.strftime("%Y-%m-%d")

# Map the control Replicate names to the same identifiers as in the single-cell data
control_mapping = {
    "Control / Control 1 / 2025-07-01": "Control / Cnt1.1 / 2025-07-01",
    "Control / Control 2 / 2025-07-01": "Control / Cnt1.2 / 2025-07-01",
    "Control / Control 1 / 2025-07-16": "Control / Cnt2.1 / 2025-07-16",
    "Control / Control 2 / 2025-07-16": "Control / Cnt2.2 / 2025-07-16",
    "Control / Control 1 / 2025-07-17": "Control / Cnt3.1 / 2025-07-17",
    "Control / Control 2 / 2025-07-17": "Control / Cnt3.2 / 2025-07-17",
    "Control / Control 1 / 2025-08-14": "Control / Cnt4.1 / 2025-08-14",
    "Control / Control 2 / 2025-08-14": "Control / Cnt4.2 / 2025-08-14",
}
facs_df["Replicate"] = facs_df["Replicate"].replace(control_mapping)
# Drop "Control / Cnt1.1 / 2025-07-01 given that it wasn't sequenced
facs_df = facs_df[facs_df["Replicate"] != "Control / Cnt1.1 / 2025-07-01"]

In [ ]:
# Get number of replicates for each Transcription factor
replicate_counts = facs_df["Replicate"].value_counts()
replicate_counts

In [ ]:
# Format FACS data
facs_df = facs_df.drop(columns=["Comments", "Sample no.", "Batch date"]).set_index("Replicate")

# Sum EEC prog. and EEC mature to get total EECs if both columns are present
facs_df["EECs"] = facs_df["EEC prog."] + facs_df["EEC mature"]
facs_df = facs_df.drop(columns=["EEC prog.", "EEC mature"])

# Rename Goblet to Goblet Cells & Entrocytes to Enterocytes, EEC prog. & EEC mature to EECs, Stem/TA Cells to Stem_TA Cells
facs_df = facs_df.rename(columns={"Goblet": "Goblet_Cells", "Entrocytes": "Enterocytes", "Stem/TA Cells": "Stem_TA_Cells"})

# Compute remainder of the percentage to 100% for each TF
facs_df["Stem_TA_Cells"] = 100 - facs_df[["Goblet_Cells", "Enterocytes", "EECs"]].sum(axis=1)

# Convert CONTROL to Control in the Transcription factor column
facs_df["Transcription factor"] = facs_df["Transcription factor"].str.replace("CONTROL", "Control")

In [ ]:
# Remove BAMBI from facs_df
facs_df = facs_df[~facs_df.index.str.contains("BAMBI")]

In [ ]:
facs_df.to_csv(tables_dir / "facs_data_formatted.csv")

In [ ]:
tables_dir / "facs_data_formatted.csv"

In [ ]:
cell_counts = (facs_df[[
    "Stem_TA_Cells",
    "Enterocytes",
    "Goblet_Cells",
    "EECs"
]] * 100).astype(int)

obs = []

for sample, row in cell_counts.iterrows():
    for cell_type, n_cells in row.items():
        for i in range(n_cells):
            obs.append({
                "sample": sample,
                "annotation": cell_type
            })

obs = pd.DataFrame(obs)
# Merge obs with facs_df to get the Transcription factor annotation
obs = obs.merge(facs_df.reset_index()[["Replicate", "Transcription factor"]], left_on="sample", right_on="Replicate", how="left")
# Rename Transcription factor to condition
obs = obs.rename(columns={"Transcription factor": "condition"})
# Drop the Replicate column
obs = obs.drop(columns=["Replicate"])

X = np.ones((len(obs), 2))

facs_adata = ad.AnnData(
    X=X,
    obs=obs,
)

facs_adata.obs_names = [
    f"cell_{i}"
    for i in range(facs_adata.n_obs)
]

facs_adata.obs["annotation"] = pd.Categorical(facs_adata.obs["annotation"], categories=cell_type_colors.keys(), ordered=True)

# Add annotation_uns to the adata
facs_adata.uns["annotation_colors"] = list(cell_type_colors.values())

# Save the anndata
facs_adata.write(anndata_dir / "facs_adata.h5ad")

In [ ]:
facs_adata = sc.read(anndata_dir / "facs_adata.h5ad")

In [ ]:
facs_adata.obs["annotation"].cat.categories

In [ ]:
facs_adata.uns["annotation_colors"]

## Submit job via sHPC

In [ ]:
%%bash
cat > run_scprop.sbatch <<'EOF'
#!/bin/bash
#SBATCH --job-name=run_scprop
#SBATCH --output=run_scprop_%j.out
#SBATCH --error=run_scprop_%j.err
#SBATCH --cpus-per-task=1
#SBATCH --mem=50G
#SBATCH --partition=batch_cpu
#SBATCH --qos=1d

source ~/.bashrc
conda activate /projects/site/pred/ihb-global/ihb-genomics-framework/conda/nf-core

WORKFLOW_DIR="/projects/site/pred/ihb-global/ihb-genomics-framework/workflows/nf-scprop"
DATA_DIR="/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/anndata_objects"
RESULTS_DIR="/projects/site/pred/ihb-g-deco/USERS/adaml9/tmp/scprop_results"
WORK_DIR="/projects/site/pred/ihb-g-deco/USERS/adaml9/tmp/scprop_work"

nextflow run "$WORKFLOW_DIR" \
    -profile local_conda \
    --conda_env /pmount/projects/site/pred/ihb-g-deco/USERS/adaml9/miniforge3/envs/pertpy \
    -work-dir "$WORK_DIR" \
    --input "${DATA_DIR}/facs_adata.h5ad" \
    --sample-key sample \
    --annotation-key annotation \
    --condition-key condition \
    --control-label Control \
    --outdir "$RESULTS_DIR" \
    -resume
EOF

sbatch run_scprop.sbatch

In [ ]:
!cp -r /projects/site/pred/ihb-g-deco/USERS/adaml9/tmp/scprop_results /projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/external_data/scprop_results

## Plot proportions

In [ ]:
# Subset to Control
facs_df_control = facs_df[facs_df.index.str.contains("Control")]
facs_proportions_control = facs_df_control[["Stem_TA_Cells", "Enterocytes", "Goblet_Cells", "EECs"]]

# Compute average proportion for each control
controls = facs_proportions_control.index.str.split(" / ").str[1].str.split(".").str[0]  # Extract Cnt1, Cnt2, Cnt3, Cnt4
facs_proportions_control["Control"] = controls
facs_proportions_control = facs_proportions_control.groupby("Control")[["Stem_TA_Cells", "Enterocytes", "Goblet_Cells", "EECs"]].mean().T

# Get the rest
facs_df_perturbation = facs_df[~facs_df.index.str.contains("Control")]

# Compute average proportions for each TF across replicates
facs_proportions_perturbation = facs_df_perturbation.groupby("Transcription factor")[["Stem_TA_Cells", "Enterocytes", "Goblet_Cells", "EECs"]].mean().T

In [ ]:
import marsilea as ma
import marsilea.plotter as mp

wb1 = ma.ClusterBoard(facs_proportions_control, width=2, height=3, margin=0.2)
wb1.add_layer(mp.StackBar(facs_proportions_control, colors=cell_type_colors))
wb1.add_bottom(mp.Labels(labels=facs_proportions_control.columns, rotation=90, fontsize=8))
#wb1.add_dendrogram("top", method="ward", pad=0, show=False)
wb1.add_legends()

with plt.rc_context({"axes.grid": False}):
    wb1.render()
    plt.savefig(figures_dir / "FACS_control_proportions.pdf", bbox_inches="tight")

In [ ]:
# Now plot the perturbation data
wb2 = ma.ClusterBoard(facs_proportions_perturbation, width=10, height=3, margin=0.2)
wb2.add_layer(mp.StackBar(facs_proportions_perturbation, colors=cell_type_colors))
wb2.add_bottom(mp.Labels(labels=facs_proportions_perturbation.columns, rotation=90, fontsize=8))
wb2.add_dendrogram("top", method="ward", pad=0, show=False)
wb2.add_legends()

with plt.rc_context({"axes.grid": False}):
    wb2.render()
    plt.savefig(figures_dir / "FACS_perturbation_proportions.pdf", bbox_inches="tight")

In [ ]:
figures_dir

## Load single-cell data

In [ ]:
# Define color palette for each cell type
cell_type_colors = {
    # EEC progenitors
    "EEC Progenitors": "#9467bd",   # purple
    
    # EEC subtypes 
    "EC Cells": "#d62728",            # strong red
    "X Cells": "#e377c2",             # magenta
    "D Cells": "#17becf",             # cyan / turquoise
    "I/N Cells": "#7f7f7f",           # dark grey
    "K Cells": "#bcbd22",             # olive 
}

In [ ]:
df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "all_eecs_pre" / "eecs_annotation_control_cell_prop_per_sample.csv", index_col=0) 

In [ ]:
df

In [ ]:
df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "all_eecs_pre" / "eecs_annotation_control_cell_prop_per_sample.csv", index_col=0) 
# Subset to non-empty columns i.e. only Controls
df = df.dropna(axis=1, how='all')
# Pool controls for each day i.e. Cnt2_1, Cnt2_2 -> Cnt2
df.columns = df.columns.str.replace(r'(_\d+)$', '', regex=True)
# Average proportions across control replicates for each day
df = df.groupby(df.columns, axis=1).mean()

# Plot as stacked barplot using marsilea
h = ma.ClusterBoard(df, width=3, height=1.5)
h.add_layer(mp.StackBar(df, label="Cell Type", colors=cell_type_colors))
h.add_bottom(mp.Labels(df.columns), size=1, pad=0.1)
h.add_legends()

with plt.rc_context({'axes.grid': False}):
    h.render()
    plt.savefig(sc.settings.figdir / "all_eecs_control_cell_proportions.pdf", bbox_inches='tight')

In [ ]:

df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "all_eecs_pre" / "eecs_annotation_cell_prop_per_condition.csv", index_col=0) 
# Remove Bambi
df = df.drop(columns=["Bambi", "Control"])
# Subset to non-empty columns
df = df.dropna(axis=1, how='all')
print(df.shape)

# Plot as stacked barplot using marsilea
h = ma.ClusterBoard(df, width=10, height=1.5)
h.add_layer(mp.StackBar(df, label="Cell Type", colors=cell_type_colors))
h.add_bottom(mp.Labels(df.columns), size=1, pad=0.1)
h.add_dendrogram("top", method="ward", pad=0, show=False)
h.add_legends()

with plt.rc_context({'axes.grid': False}):
    h.render()
    plt.savefig(sc.settings.figdir / "all_eecs_perturb_cell_proportions.pdf", bbox_inches='tight')